# 15.5 — Gingival detector with an ROI classification filter

Compare the saved YOLOv8n predictions with the same boxes filtered by the existing R1 ResNet18. Evaluate only on the locked D2 validation split.

## Verify inputs and load the existing classifier

Reuse the YOLOv8n v4 outputs from notebook 15.3 at confidence ≥0.25. The ROI classifier predicts the three original D2 grades. Remove a box when its highest-scoring grade is 0.

In [1]:
from pathlib import Path
from io import BytesIO
import json
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torchvision import transforms
from torchvision.models import resnet18
from dentalselfie.acquire import digest
from dentalselfie.detection_audit import box_iou, match_boxes
from dentalselfie.roi import padded_view, view_bounds

root = Path.cwd()
while not (root / 'pyproject.toml').exists():
    if root == root.parent:
        raise RuntimeError('Project root not found')
    root = root.parent
report = root / 'docs/reports/gingivitis_binary'
output = report / 'roi_filter'
output.mkdir(exist_ok=True)
split = root / 'data/manifests/gum.splits.csv'
lock = json.loads((split.parent / 'gum.lock.json').read_text())
assert digest(split) == lock['splits_sha256']
parents = pd.read_csv(split).query("split == 'val'").set_index('sample_id')
assert len(parents) == 160 and parents.index.is_unique
predictions = pd.read_csv(report / 'audit_predictions.csv').query("model == 'YOLOv8n' and confidence >= 0.25").copy()
assert len(predictions) == 2060 and set(predictions.sample_id) <= set(parents.index)
checkpoint = root / 'data/runs/ROI_09_1_v1/dentalselfie-09-1-resolution-control/full/R1_raw_CE/best.pt'
model = resnet18(weights=None, num_classes=3)
model.load_state_dict(torch.load(checkpoint, map_location='cpu', weights_only=True))
model.eval()
torch.set_num_threads(min(4, torch.get_num_threads()))
roi_config = json.loads((root / 'src/configs/roi.json').read_text())
assert roi_config['size'] == 224 and roi_config['classes'][0] == 'no_visible_inflammation'
transform = transforms.Compose([
    transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])
coords = ['x1','y1','x2','y2']
print('R1 SHA-256:', digest(checkpoint))
print('Validation images:', len(parents), '| Candidate boxes:', len(predictions))

R1 SHA-256: 4f9b1e67ef03bdbf7fd535ffd59b83085fa7470bb3b96627ce6b9555f87fc4e0
Validation images: 160 | Candidate boxes: 2060


## Classify the predicted regions

Project normalized detector coordinates onto the hash-verified raw source photo. Apply the R1 crop convention, gray padding, bilinear resize and JPEG encoding before ImageNet normalization. The detector confidence and box coordinates remain the inputs to the comparison.

In [2]:
probabilities = []
with torch.inference_mode():
    for sample, group in predictions.groupby('sample_id', sort=False):
        parent = parents.loc[sample]
        path = root / parent.source_image
        assert digest(path) == parent.source_sha256
        with Image.open(path) as photo:
            assert photo.getexif().get(274, 1) in (0, 1)
            image = photo.convert('RGB')
        assert image.size == (parent.source_width, parent.source_height)
        crops = []
        scale = np.array([image.width, image.height] * 2)
        for box in group[coords].to_numpy():
            crop, _ = padded_view(image, view_bounds(box * scale), roi_config['size'], roi_config['padding_rgb'])
            with BytesIO() as buffer:
                crop.save(buffer, format='JPEG', quality=95, subsampling=0)
                buffer.seek(0)
                with Image.open(buffer) as decoded:
                    crops.append(transform(decoded.convert('RGB')))
        scores = model(torch.stack(crops)).softmax(1).numpy()
        probabilities.extend((index, *score) for index, score in zip(group.index, scores))
classified = pd.DataFrame(probabilities, columns=['index','p_grade0','p_mild','p_marked']).set_index('index')
assert classified.index.is_unique and classified.index.symmetric_difference(predictions.index).empty
predictions = predictions.join(classified)
predictions['roi_grade'] = predictions[['p_grade0','p_mild','p_marked']].to_numpy().argmax(axis=1)
predictions['retained'] = predictions.roi_grade != 0
predictions.to_csv(output / 'classified_boxes.csv', index=False)
display(predictions.groupby('roi_grade').size().rename('Boxes'))

roi_grade
0      55
1    1135
2     870
Name: Boxes, dtype: int64

## Compare against detector alone

Reuse the source regions from notebook 15.3, including the single ambiguous positive box. Match each arm independently at IoU ≥0.50 using descending detector confidence. Grade-0 regions count as flagged when at least one unmatched prediction overlaps the annotation at IoU ≥0.50.

In [3]:
positive = pd.read_csv(report / 'audit_regions.csv').query("model == 'YOLOv8n' and confidence == 0.25")
negative = pd.read_csv(report / 'audit_grade0_regions.csv').query("model == 'YOLOv8n' and confidence == 0.25")
assert (len(positive), len(negative)) == (1870, 49)
results = []
image_rows = []
for method, pool in [('YOLOv8n', predictions), ('YOLOv8n + R1', predictions[predictions.retained])]:
    found = {1: 0, 2: 0, 3: 0}
    flagged = unmatched_count = 0
    for sample in parents.index:
        gt = positive[positive.sample_id == sample]
        grade0 = negative[negative.sample_id == sample]
        candidates = pool[pool.sample_id == sample]
        boxes = candidates[coords].to_numpy()
        events, _ = match_boxes(gt[coords], boxes, candidates.confidence)
        matched = [event['gt_index'] for event in events if event['gt_index'] is not None]
        for grade in gt.iloc[matched].grade:
            found[grade] += 1
        unmatched = [event['prediction_index'] for event in events if event['gt_index'] is None]
        overlaps = box_iou(boxes[unmatched], grade0[coords])
        flagged_count = int((overlaps >= 0.50).any(axis=0).sum())
        flagged += flagged_count
        unmatched_count += len(unmatched)
        image_rows.append((method, sample, len(gt), len(matched), len(grade0), flagged_count, len(unmatched)))
    results.append(dict(method=method, boxes=len(pool), positive_found=sum(found.values()),
                        mild_found=found[1], marked_found=found[2], ambiguous_found=found[3],
                        grade0_flagged=flagged, unmatched_predictions=unmatched_count))
summary = pd.DataFrame(results)
assert summary.iloc[0][['positive_found','mild_found','marked_found','grade0_flagged','unmatched_predictions']].tolist() == [1814,908,905,49,246]
summary.to_csv(output / 'comparison.csv', index=False)
pd.DataFrame(image_rows, columns=['method','sample_id','positives','matched','grade0','grade0_flagged','unmatched']).to_csv(output / 'per_image.csv', index=False)
display(summary)

,method,boxes,positive_found,mild_found,marked_found,ambiguous_found,grade0_flagged,unmatched_predictions
0,YOLOv8n,2060,1814,908,905,1,49,246
1,YOLOv8n + R1,2005,1776,875,900,1,40,229
